In [0]:
%pip install Faker
import os
import sys

import json
import pandas as pd 
from faker import Faker
import custom_providers as cp
from datetime import datetime

In [0]:
NUM_RECORDS = 5000
DELTA_DATABASE = "default"
DELTA_MODE = "overwrite"

In [0]:
def setup_faker():
    fake = Faker(["en_US"])
    
    # Personal Information
    fake.add_provider(cp.PersonNameProvider)
    fake.add_provider(cp.NationalityProvider)
    fake.add_provider(cp.EmailProvider)
    fake.add_provider(cp.TimeProvider)
    fake.add_provider(cp.AgeProvider)
    fake.add_provider(cp.DOBProvider)
    fake.add_provider(cp.Gender)
    fake.add_provider(cp.Orientation)
    fake.add_provider(cp.Title)
    fake.add_provider(cp.MaritalStatus)
    fake.add_provider(cp.Relation)

    # Identification
    fake.add_provider(cp.UsDriverLicenseProvider)
    fake.add_provider(cp.CustomSSNUSProvider)
    fake.add_provider(cp.PassportProvider)
    fake.add_provider(cp.DocumentIDProvider)
    fake.add_provider(cp.RegionIDProvider)
    fake.add_provider(cp.UserIDProvider)

    # Financial
    fake.add_provider(cp.CVV)
    fake.add_provider(cp.CustomCreditCardProvider)
    fake.add_provider(cp.BankUSProvider)
    fake.add_provider(cp.MoneyProvider)
    fake.add_provider(cp.CreditCardTransactionTypeProvider)
    fake.add_provider(cp.CreditCardTransactionDetailProvider)
    fake.add_provider(cp.NonCreditTransactionTypeProvider)
    fake.add_provider(cp.NonCreditTransactionDetailProvider)
    fake.add_provider(cp.ExpiryDateProvider)
    fake.add_provider(cp.CardTypeProvider)
    fake.add_provider(cp.DueDateProvider)
    fake.add_provider(cp.NonCreditCardProvider)
    fake.add_provider(cp.CardOperationProvider)
    fake.add_provider(cp.CardSalutationProvider)
    fake.add_provider(cp.CardCustomerSalutationProvider)
    fake.add_provider(cp.TransactionIDProvider)

    # Location & Contact
    fake.add_provider(cp.AddressUSProviderFull)
    fake.add_provider(cp.PhoneNumberProviderNew)
    fake.add_provider(cp.IpAddressProvider)

    # Social & Misc
    fake.add_provider(cp.USPoliticalParties)
    fake.add_provider(cp.IdeologyProvider)
    fake.add_provider(cp.VulnerableCommunity)
    fake.add_provider(cp.ReligionProvider)
    fake.add_provider(cp.JobProvider)
    fake.add_provider(cp.SkillSet)
    fake.add_provider(cp.RandomStarter)
    fake.add_provider(cp.AlphanumericProvider)
    fake.add_provider(cp.Separator)
    fake.add_provider(cp.SessionIDProvider)
    fake.add_provider(cp.FilenameProvider)
    fake.add_provider(cp.ProfileIDProvider)
    fake.add_provider(cp.TimestampProvider)
    fake.add_provider(cp.CustomURLProvider)
    fake.add_provider(cp.TeamsProvider)
    fake.add_provider(cp.LawProvider)
    fake.add_provider(cp.ProductsProvider)
    
    return fake

In [0]:
def generate_credit_card_transaction(fake):
    # Generate a credit card transaction record
    name = fake.person_name()
    card_number = fake.credit_card_number(with_separator=True, random_spaces=True)

    roc = fake.random_int(min=100000, max=999999)
    uri = 'CREDIT CARD' if fake.random_int(0,1) == 1 else 'CC'
    transaction_string = f"/ROC/{roc}/URI/{uri} PAYMENT FOR {name.upper()} {card_number}"

    return {
        "transaction_format": transaction_string,
        "transaction_id": fake.transaction_id(),
        "customer_name": name,
        "card_number": card_number,
        "card_type": fake.card_type(),
        "cvv": fake.cvv(),
        "expiry_date": fake.expiry_date(),
        "transaction_type": fake.transaction_type(),
        "transaction_detail": fake.transaction_detail(),
        "amount": fake.currency_amount(),
        "timestamp": fake.timestamp("datetime"),
        "merchant_id": fake.alphanumeric(12, uppercase=True),
    }

In [0]:
def generate_complete_customer_profile(fake):
    """Generate a complete customer profile with all PII."""
    name = fake.person_name()
    address = fake.full_address()

    return {
        # Personal
        "profile_id": fake.profile_id(),
        "title": fake.title(),
        "full_name": name,
        "gender": fake.gender(),
        "date_of_birth": fake.date_of_birth(),
        "age": fake.age(),
        "marital_status": fake.marital_status(),
        "nationality": fake.nationality(),
        
        # Contact
        "email": fake.email_address(name),
        "phone": fake.phone_number(),
        "mobile": fake.mobile_number(),
        "address": address,

        # Identification
        "ssn": fake.ssn(),
        "driver_license": fake.driver_license(),
        "passport": fake.passport_number(),
        "passport_expiry": fake.passport_expiry(),

        # Financial
        "credit_card_number": fake.credit_card_number(),
        "cvv": fake.cvv(),
        "expiry_date": fake.expiry_date(),
        "bank_name": fake.bank_name(),
        "account_number": fake.account_number(),
        "routing_number": fake.routing_number(),

        # Employment
        "job_title": fake.job_title(),
        "employment_status": fake.employment_status(),
        "skills": fake.skills_list(),
        "team": fake.team(),

        # Social
        "religion": fake.religion(),
        "political_party": fake.political_party(),
        "ideology": fake.ideology(),

        # Technical
        "username": fake.username(),
        "user_id": fake.user_id(),
        "ip_address": fake.ipv4(),
        "session_id": fake.session_id(),
        "profile_url": fake.url(with_path=True),
    }

In [0]:
def generate_non_credit_transaction(fake):
    """Generate a non-credit card transaction."""
    name = fake.person_name()

    return {
        "transaction_id": fake.transaction_id(prefix="ACH"),
        "customer_name": name,
        "transaction_type": fake.transaction_type(),
        "transaction_detail": fake.transaction_detail(),
        "amount": fake.currency_amount(),
        "timestamp": fake.timestamp("datetime"),
        "bank_name": fake.bank_name(),
        "account_number": fake.account_number(),
    }

In [0]:
def write_to_delta_table(records, table_name, database="default", mode="overwrite"):
    """
    Convert a list of dicts to a Spark DataFrame and write to a Delta table.

    Args:
        records:    List of dicts from any generator function.
        table_name: Unqualified table name.
        database:   Target database / schema.
        mode:       "overwrite" or "append".
    """
    qualified_name = f"{database}.{table_name}"
    print(f"Writing {len(records):,} records → '{qualified_name}' (mode={mode}) ...")

    df_pandas = pd.DataFrame(records)

    # Serialize any list / dict columns to JSON strings so Spark can infer schema.
    for col in df_pandas.columns:
        if df_pandas[col].dtype == object:
            sample = df_pandas[col].dropna()
            if len(sample) > 0 and isinstance(sample.iloc[0], (list, dict)):
                df_pandas[col] = df_pandas[col].apply(
                    lambda v: json.dumps(v) if v is not None else None
                )

    (
        spark.createDataFrame(df_pandas)
        .write.format("delta")
        .mode(mode)
        .saveAsTable(qualified_name)
    )
    print(f"  ✓ '{qualified_name}' written successfully.")


print(f"Generating {NUM_RECORDS:,} records per dataset ...\n")

fake = setup_faker()

credit_card_transactions = [generate_credit_card_transaction(fake) for _ in range(NUM_RECORDS)]
customer_profiles         = [generate_complete_customer_profile(fake) for _ in range(NUM_RECORDS)]
non_credit_transactions   = [generate_non_credit_transaction(fake) for _ in range(NUM_RECORDS)]

write_to_delta_table(credit_card_transactions, "credit_card_transactions", DELTA_DATABASE, DELTA_MODE)
write_to_delta_table(customer_profiles,        "customer_profiles",        DELTA_DATABASE, DELTA_MODE)
write_to_delta_table(non_credit_transactions,  "non_credit_transactions",  DELTA_DATABASE, DELTA_MODE)

print(f"\n✓ Done - all tables written to '{DELTA_DATABASE}'.")

for table in ["credit_card_transactions", "customer_profiles", "non_credit_transactions"]:
    count = spark.table(f"{DELTA_DATABASE}.{table}").count()
    print(f"{DELTA_DATABASE}.{table}: {count:,} rows")